# Toxic Gaming Chat Classifier - Colab Training

**By Alex You**

This notebook uses Colab as a GPU runtime while keeping the project code in Python modules.

Link to Colab: https://colab.research.google.com/drive/1f8KZVrd-1DDIg9hbUXw2xdMJRg0mT_ji?usp=sharing

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

CUDA available: True
GPU: Tesla T4
Fri Jul 10 22:53:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

## 2. Mount Google Drive

Set `PROJECT_DIR` to the folder that contains this repository in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your repo is stored somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/APS360/Final_Project'
%cd $PROJECT_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/APS360/Final_Project


## 3. Install Dependencies

In [ ]:
%pip install -r requirements.txt

## 4. Verify Or Prepare Data

The training scripts expect `data/tribunal/tribunal_chat_100k_balanced.csv` with `text,label` columns. If only raw `chatlogs.csv` is present, run the prep script here.

In [ ]:
from pathlib import Path
import subprocess

import pandas as pd

prepared_path = Path('data/tribunal/tribunal_chat_100k_balanced.csv')
raw_path = Path('data/tribunal/chatlogs.csv')

if not prepared_path.exists():
    if not raw_path.exists():
        raise FileNotFoundError(
            'Upload tribunal_chat_100k_balanced.csv or raw chatlogs.csv to data/tribunal/.'
        )
    # Prefer subprocess over !python so this cell stays valid Python under control flow.
    subprocess.run(['python', 'prepare_tribunal.py'], check=True)

df = pd.read_csv(prepared_path)
print(df.head()) # Print first  5 rows
print(df['label'].value_counts().sort_index())

                        text  label
0                        lol      1
1                        brb      1
2                        man      0
3  what league on your main?      1
4          im struggling mid      1
label
0    50000
1    50000
Name: count, dtype: int64


## 5. Train LSTM

In [ ]:
!python train.py

Device: cuda
Epoch 01/10 | train loss 0.6886 acc 0.537 | val loss 0.6856 acc 0.547  <- saved
Epoch 02/10 | train loss 0.6794 acc 0.570 | val loss 0.6858 acc 0.550
Epoch 03/10 | train loss 0.6646 acc 0.597 | val loss 0.6941 acc 0.551
Epoch 04/10 | train loss 0.6392 acc 0.628 | val loss 0.7138 acc 0.548
Epoch 05/10 | train loss 0.6048 acc 0.656 | val loss 0.7367 acc 0.549
Epoch 06/10 | train loss 0.5648 acc 0.684 | val loss 0.7900 acc 0.547
Epoch 07/10 | train loss 0.5285 acc 0.706 | val loss 0.8743 acc 0.544
Epoch 08/10 | train loss 0.4979 acc 0.724 | val loss 0.9291 acc 0.542
Epoch 09/10 | train loss 0.4736 acc 0.738 | val loss 0.9868 acc 0.544
Epoch 10/10 | train loss 0.4553 acc 0.749 | val loss 1.0391 acc 0.542

Best model test accuracy: 0.545


## 6. Run Baseline

In [ ]:
!python baseline.py

Baseline (TF-IDF + LinearSVC) test accuracy: 0.552

              precision    recall  f1-score   support

        safe       0.55      0.59      0.57      7500
       toxic       0.56      0.51      0.53      7500

    accuracy                           0.55     15000
   macro avg       0.55      0.55      0.55     15000
weighted avg       0.55      0.55      0.55     15000



## 7. Save Checkpoint To Drive

The repository already lives in Drive if `PROJECT_DIR` points there, but this also copies the best weights into an explicit Colab artifacts folder.

In [ ]:
from pathlib import Path
import shutil

# Keep artifacts next to the project so the path matches PROJECT_DIR from cell 4.
artifact_dir = Path(PROJECT_DIR) / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

src = Path('checkpoints/best_model.pt')
dst = artifact_dir / 'best_model.pt'
if src.exists():
    shutil.copy2(src, dst)
    print(f'Copied checkpoint to {dst}')
else:
    print('No checkpoint found. Run training first.')

Copied checkpoint to /content/drive/MyDrive/APS360/Final_Project/artifacts/best_model.pt
